# 03 — Leakage-safe splits + reference detector (O3, part 1)

**Settings:** Accelerator **GPU T4 x2** (DDP is used automatically), Internet **On**.
**Input:** *Underwater Domain in ODverse33* (skycol).

**Kaggle tips**
- Re-running cell 1 always pulls the latest code from GitHub. You never need to re-import this notebook.
- `/kaggle/working` is wiped when a session ends unless you commit (**Save Version → Save & Run All**)
  or enable **Settings → Persistence → Files**.

In [ ]:
!test -d /kaggle/working/repo || git clone -q https://github.com/Banasri007/UW-detection-aware-iqa.git /kaggle/working/repo
%run /kaggle/working/repo/scripts/kaggle_setup.py detect

In [ ]:
!python scripts/build_manifests.py --only ruod duo

### 1. Duplicate threshold
Notebook 02 showed that pairs at d=20 are still the same video scene, so we search wider (32/256) and look at the band just above 20. Hashes are cached, so this takes ~3 min if notebook 02's `results/dedup` is present, ~10 min otherwise.

In [ ]:
!python scripts/find_duplicates.py --datasets ruod duo --max-dist 32

In [ ]:
%run scripts/show_pairs.py --min-dist 21 --max-dist 26 --leaky-only --n 6

In [ ]:
%run scripts/show_pairs.py --min-dist 27 --max-dist 32 --leaky-only --n 6

If the 27–32 pairs are still the same scene, keep 32. If they are different scenes, re-run the duplicate cell with the largest distance that still looked like a duplicate.

### 2. Leakage-safe splits

In [ ]:
!python scripts/make_splits.py

### 3. Reference detector (YOLO11-s on raw RUOD `det_train`)
Run the smoke test first (~5 min). For the full run, use **Save Version → Save & Run All**: roughly 3–5 h on T4 x2. If it times out, attach this notebook's output as input, copy `results/detector` back to `/kaggle/working/results/`, and re-run: training resumes from `last.pt`.

In [ ]:
# Smoke test: 5% of det_train, 2 epochs
!python scripts/train_detector.py --epochs 2 --fraction 0.05 --name smoke

In [ ]:
# Full training + held-out evaluation on RUOD pred_test and duplicate-free DUO
!python scripts/train_detector.py